# 加阈值44，判断是否标记故障

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # For reproducibility

# Read the data from the CSV file
file_path = r"F:\OneDrive - CentraleSupelec\Courses\data_experiment_collect\dataset_collect_N5\val_data.csv"
data = pd.read_csv(file_path)

# Extract the specific column for processing
if "motor6_temperature" not in data.columns:
    raise ValueError("The column 'motor6_temperature' is not in the provided CSV file.")
motor6_temperature = data["motor6_temperature"].to_numpy()
motor6_temperature_true = data["motor6_temperature"].to_numpy()

def insert_faults(sequence, num_faults=400, fault_length_range=(30, 50), temp_increase_range=(6, 10), threshold=None):
    """
    Insert faults into a sequence and optionally mark values as faults based on a threshold.

    Parameters:
    - sequence: Original data sequence.
    - num_faults: Number of faults to inject.
    - fault_length_range: Range of fault lengths (min, max).
    - temp_increase_range: Range of temperature increases (min, max).
    - threshold: If provided, mark values as faults only if they exceed the threshold.

    Returns:
    - modified_sequence: Sequence with injected faults.
    - labels: Array where faults are marked as 1.
    - fault_indices: List of fault start indices.
    """
    modified_sequence = sequence.astype(float)
    length = len(sequence)
    labels = np.zeros(length, dtype=int)
    fault_indices = []

    for _ in range(num_faults):
        # Find a suitable start index
        for _ in range(100):
            start_idx = np.random.randint(0, length - fault_length_range[1])
            if all(abs(start_idx - idx) >= fault_length_range[1] for idx in fault_indices):
                break
        else:
            continue

        # Fault properties
        fault_length = np.random.randint(*fault_length_range)
        temp_increase = np.random.uniform(*temp_increase_range)

        # Fault trend: rise, plateau, fall
        rise_length = int(fault_length * 0.3)
        plateau_length = int(fault_length * 0.3)
        fall_length = fault_length - rise_length - plateau_length

        rise = np.linspace(0, temp_increase, rise_length)
        plateau = np.full(plateau_length, temp_increase)
        fall = np.linspace(temp_increase, 0, fall_length)

        fault_trend = np.concatenate([rise, plateau, fall])

        # Apply fault
        end_idx = min(start_idx + fault_length, length)
        modified_sequence[start_idx:end_idx] += fault_trend[:end_idx - start_idx]

        # Mark faults
        if threshold is None:
            labels[start_idx:end_idx] = 1
        else:
            for i in range(start_idx, end_idx):
                if modified_sequence[i] > threshold:
                    labels[i] = 1

        fault_indices.append(start_idx)

    # Ensure faults are sorted from smallest to largest
    fault_indices.sort()
    return modified_sequence, labels, fault_indices

# Choose the mode: Direct marking or threshold-based marking
marking_mode = "threshold"  # "direct" or "threshold"
threshold_value = 44 if marking_mode == "threshold" else None

# Inject faults and mark
modified_sequence, labels, fault_indices = insert_faults(
    motor6_temperature,
    num_faults=20,
    fault_length_range=(30, 50),
    temp_increase_range=(6, 10),
    threshold=threshold_value
)

# Visualization
plt.figure(figsize=(14, 6))
plt.plot(modified_sequence, label="Faulty motor6_temperature", color="red" if marking_mode == "direct" else "red", alpha=0.9)
# plt.plot(motor6_temperature_true[500:2000], label="Faulty motor6_temperature", color="red" if marking_mode == "direct" else "blue", alpha=0.9)


# Add fault markers and annotate with fault numbers
last_label_y = modified_sequence.max() + 2  # Starting Y-position for fault labels
for idx, fault in enumerate(fault_indices):
    plt.axvline(x=fault, color='green', linestyle='--', alpha=0.5)
    label_y = last_label_y-0.5 # - 0.5 * (idx % 3)  # Avoid overlap by alternating label heights
    plt.text(fault, label_y, f"Fault {idx + 1}", rotation=90, verticalalignment='bottom', fontsize=10)

# Add title and legend
plt.title(f"Faults Injected ({'Direct Marking' if marking_mode == 'direct' else f'Threshold > {threshold_value}'})")
plt.xlabel("Time Steps")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid()
plt.show()

# Save modified data
data["motor6_temperature"] = modified_sequence
data["label"] = labels

output_path = r"F:\OneDrive - CentraleSupelec\Courses\data_experiment_collect\dataset_collect_N5"
output_file = f"{output_path}/val_data_with_faults_{'direct' if marking_mode == 'direct' else f'threshold_{threshold_value}'}.csv"
data.to_csv(output_file, index=False)

print("Modified file saved to:", output_file)


# 三种方法注入故障

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # For reproducibility

# Read the data from the CSV file
file_path = r"F:\OneDrive - CentraleSupelec\PhD\Paper-Fault_detection\codes\data\target_test_data.csv"
data = pd.read_csv(file_path)

# Extract the specific column for processing
if "motor6_temperature" not in data.columns:
    raise ValueError("The column 'motor6_temperature' is not in the provided CSV file.")
motor6_temperature = data["motor6_temperature"].to_numpy()
motor6_temperature_true = data["motor6_temperature"].to_numpy()

def insert_faults(sequence, num_faults=20, fault_length_range=(30, 50), temp_increase_range=(6, 10), threshold_increase=None, marking_mode="direct"):
    """
    Insert faults into a sequence and mark values as faults based on different marking modes.

    Parameters:
    - sequence: Original data sequence.
    - num_faults: Number of faults to inject.
    - fault_length_range: Range of fault lengths (min, max).
    - temp_increase_range: Range of temperature increases (min, max).
    - threshold_increase: If provided, mark values as faults only if they exceed the original value by the threshold.
    - marking_mode: "direct", "threshold", or "custom_threshold" for different marking methods.

    Returns:
    - modified_sequence: Sequence with injected faults.
    - labels: Array where faults are marked as 1.
    - fault_indices: List of fault start indices.
    """
    modified_sequence = sequence.astype(float)
    length = len(sequence)
    labels = np.zeros(length, dtype=int)
    fault_indices = []

    for _ in range(num_faults):
        # Find a suitable start index
        for _ in range(100):
            start_idx = np.random.randint(0, length - fault_length_range[1])
            if all(abs(start_idx - idx) >= fault_length_range[1] for idx in fault_indices):
                break
        else:
            continue

        # Fault properties
        fault_length = np.random.randint(*fault_length_range)
        temp_increase = np.random.uniform(*temp_increase_range)

        # Fault trend: rise, plateau, fall
        rise_length = int(fault_length * 0.3)
        plateau_length = int(fault_length * 0.3)
        fall_length = fault_length - rise_length - plateau_length

        rise = np.linspace(0, temp_increase, rise_length)
        plateau = np.full(plateau_length, temp_increase)
        fall = np.linspace(temp_increase, 0, fall_length)

        fault_trend = np.concatenate([rise, plateau, fall])

        # Apply fault
        end_idx = min(start_idx + fault_length, length)
        modified_sequence[start_idx:end_idx] += fault_trend[:end_idx - start_idx]

        # Mark faults
        if marking_mode == "direct":
            labels[start_idx:end_idx] = 1
        elif marking_mode == "threshold":
            for i in range(start_idx, end_idx):
                if modified_sequence[i] > sequence[i] + threshold_increase:
                    labels[i] = 1
        elif marking_mode == "custom_threshold":
            for i in range(start_idx, end_idx):
                if modified_sequence[i] > 44:  # Example of a custom threshold
                    labels[i] = 1

        fault_indices.append(start_idx)

    # Ensure faults are sorted from smallest to largest
    fault_indices.sort()
    return modified_sequence, labels, fault_indices

# Choose the mode: Direct marking, dynamic threshold, or custom threshold
marking_mode = "threshold"  # Options: "direct", "threshold", "custom_threshold"
threshold_increase = 1 if marking_mode == "threshold" else None
num_faults = 20

# Inject faults and mark
modified_sequence, labels, fault_indices = insert_faults(
    motor6_temperature,
    num_faults=num_faults,
    fault_length_range=(30, 50),
    temp_increase_range=(3, 8),
    threshold_increase=threshold_increase,
    marking_mode=marking_mode
)

# Visualization
plt.figure(figsize=(14, 6))
plt.plot(modified_sequence, label="Faulty motor6_temperature", color="red", alpha=0.9)
plt.plot(motor6_temperature_true, label="Original motor6_temperature", color="blue", alpha=0.6)

# Add fault markers and annotate with fault numbers
last_label_y = modified_sequence.max() + 2  # Starting Y-position for fault labels
for idx, fault in enumerate(fault_indices):
    plt.axvline(x=fault, color='green', linestyle='--', alpha=0.5)
    label_y = last_label_y - 0.5  # Avoid overlap by alternating label heights
    plt.text(fault, label_y, f"Fault {idx + 1}", rotation=90, verticalalignment='bottom', fontsize=10)

# Add title and legend
plt.title(f"Faults Injected (Mode: {marking_mode}{' | Threshold: Original + ' + str(threshold_increase) + '°C' if marking_mode == 'threshold' else ''})")
plt.xlabel("Time Steps")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid()
plt.show()

# Save modified data
data["motor6_temperature"] = modified_sequence
data["label"] = labels

output_path = r"F:\OneDrive - CentraleSupelec\PhD\Paper-Fault_detection\codes\data"
output_file = f"{output_path}/target_test_data_with_faults{str(num_faults)}_{marking_mode}{str(threshold_increase) if marking_mode == 'threshold' else ''}.csv"
data.to_csv(output_file, index=False)

print("Modified file saved to:", output_file)

# 最新的，包含了文件名缩写

文件名更加紧凑，包含：

故障数量 (test_faults20)
故障模式 (D, T, CT)
阈值 (如果适用) (_Th1)
故障长度范围 (_FL30-50)
温度增加范围 (_TI3-8)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

random_seed = 4
np.random.seed(random_seed)  # For reproducibility

# Read the data from the CSV file
file_path = r"F:\OneDrive - CentraleSupelec\PhD\Paper-Fault_detection\codes\data\target_test_data_b1.csv"
data = pd.read_csv(file_path)

# Extract the specific column for processing
if "motor6_temperature" not in data.columns:
    raise ValueError("The column 'motor6_temperature' is not in the provided CSV file.")

motor6_temperature = data["motor6_temperature"].to_numpy()
motor6_temperature_true = data["motor6_temperature"].to_numpy()

def insert_faults(sequence, num_faults=20, fault_length_range=(30, 50), temp_increase_range=(6, 10), threshold_increase=None, marking_mode="direct"):
    """
    Insert faults into a sequence and mark values as faults based on different marking modes.

    Parameters:
    - sequence: Original data sequence.
    - num_faults: Number of faults to inject.
    - fault_length_range: Range of fault lengths (min, max).
    - temp_increase_range: Range of temperature increases (min, max).
    - threshold_increase: If provided, mark values as faults only if they exceed the original value by the threshold.
    - marking_mode: "direct", "threshold", or "custom_threshold" for different marking methods.

    Returns:
    - modified_sequence: Sequence with injected faults.
    - labels: Array where faults are marked as 1.
    - fault_indices: List of fault start indices.
    """
    modified_sequence = sequence.astype(float)
    length = len(sequence)
    labels = np.zeros(length, dtype=int)
    fault_indices = []

    for _ in range(num_faults):
        # Find a suitable start index
        for _ in range(100):
            start_idx = np.random.randint(0, length - fault_length_range[1])
            if all(abs(start_idx - idx) >= fault_length_range[1] for idx in fault_indices):
                break
        else:
            continue

        # Fault properties
        fault_length = np.random.randint(*fault_length_range)
        temp_increase = np.random.uniform(*temp_increase_range)

        # Fault trend: rise, plateau, fall
        rise_length = int(fault_length * 0.3)
        plateau_length = int(fault_length * 0.3)
        fall_length = fault_length - rise_length - plateau_length

        rise = np.linspace(0, temp_increase, rise_length)
        plateau = np.full(plateau_length, temp_increase)
        fall = np.linspace(temp_increase, 0, fall_length)

        fault_trend = np.concatenate([rise, plateau, fall])

        # Apply fault
        end_idx = min(start_idx + fault_length, length)
        modified_sequence[start_idx:end_idx] += fault_trend[:end_idx - start_idx]

        # Mark faults
        if marking_mode == "direct":
            labels[start_idx:end_idx] = 1
        elif marking_mode == "threshold":
            for i in range(start_idx, end_idx):
                if modified_sequence[i] > sequence[i] + threshold_increase:
                    labels[i] = 1
        elif marking_mode == "custom_threshold":
            for i in range(start_idx, end_idx):
                if modified_sequence[i] > 44:  # Example of a custom threshold
                    labels[i] = 1

        fault_indices.append(start_idx)

    # Ensure faults are sorted from smallest to largest
    fault_indices.sort()
    return modified_sequence, labels, fault_indices

# Choose the mode: Direct marking, dynamic threshold, or custom threshold
marking_mode = "threshold"  # Options: "direct", "threshold", "custom_threshold"
threshold_increase = 1 if marking_mode == "threshold" else None
num_faults = 5
fault_length_range = (30, 50)
temp_increase_range = (6, 10)

# Inject faults and mark
modified_sequence, labels, fault_indices = insert_faults(
    motor6_temperature,
    num_faults=num_faults,
    fault_length_range=fault_length_range,
    temp_increase_range=temp_increase_range,
    threshold_increase=threshold_increase,
    marking_mode=marking_mode
)

# Visualization
plt.figure(figsize=(14, 6))
plt.plot(modified_sequence, label="Faulty motor6_temperature", color="red", alpha=0.9)
plt.plot(motor6_temperature_true, label="Original motor6_temperature", color="blue", alpha=0.6)

# Add fault markers and annotate with fault numbers
last_label_y = modified_sequence.max() + 2  # Starting Y-position for fault labels
for idx, fault in enumerate(fault_indices):
    plt.axvline(x=fault, color='green', linestyle='--', alpha=0.5)
    label_y = last_label_y - 0.5  # Avoid overlap by alternating label heights
    plt.text(fault, label_y, f"Fault {idx + 1}", rotation=90, verticalalignment='bottom', fontsize=10)

# Add title and legend
plt.title(f"Faults Injected (Mode: {marking_mode}{' | Threshold: Original + ' + str(threshold_increase) + '°C' if marking_mode == 'threshold' else ''})")
plt.xlabel("Time Steps")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid()
plt.show()

# File name compression strategy
mode_abbr = {"direct": "D", "threshold": "T", "custom_threshold": "CT"}
marking_short = mode_abbr.get(marking_mode, "U")  # 默认 "U" 表示未知模式
threshold_str = f"_Th{threshold_increase}" if marking_mode == "threshold" else ""
fault_length_str = f"_FL{fault_length_range[0]}-{fault_length_range[1]}"
temp_increase_str = f"_TI{temp_increase_range[0]}-{temp_increase_range[1]}"

# Construct compressed filename
output_path = r"F:\OneDrive - CentraleSupelec\PhD\Paper-Fault_detection\codes\data"
output_file = f"{output_path}/target_test_faults{num_faults}_{marking_short}{threshold_str}{fault_length_str}{temp_increase_str}_seed{random_seed}_b1.csv"

# Save modified data
data["motor6_temperature"] = modified_sequence
data["label"] = labels
data.to_csv(output_file, index=False)

print("Modified file saved to:", output_file)
